In [1]:
import glob
import os
from pathlib import Path
import json
video_paths = glob.glob(os.path.join("/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie", "**/*.mp4"), recursive=True)
y = []
for path in Path(r"/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie").rglob('*.json'):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        y.append(data['emotion'])
        
print(set(y))

{'amusement', 'disgust', 'contentment', 'sadness', 'awe', 'excitement', 'anger', 'fear'}


In [2]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(video_paths, y, train_size = 0.8, random_state = 42, stratify = y)
print(len(X_train))

10604


In [3]:
video_paths[:5], y[:5]

(['/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie/video/00114/00114_clip_017.mp4',
  '/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie/video/00114/00114_clip_006.mp4',
  '/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie/video/00114/00114_clip_032.mp4',
  '/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie/video/00114/00114_clip_007.mp4',
  '/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie/video/00114/00114_clip_036.mp4'],
 ['awe', 'awe', 'excitement', 'excitement', 'awe'])

In [10]:
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification, TrainingArguments, Trainer
import torch
from datasets import Dataset

processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
model = VideoMAEForVideoClassification.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics",
                                                       attn_implementation="sdpa",
                                                       num_labels = 8,
                                                       ignore_mismatched_sizes=True
                                                      )


train_dataset = Dataset.from_dict({
    "video_path": X_train,
    "label": y_train
})
train_dataset = train_dataset.class_encode_column("label")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400, 768]) vs model:torch.Size([8, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400]) vs model:torch.Size([8])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Casting to class labels:   0%|          | 0/10604 [00:00<?, ? examples/s]

In [11]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

def video_preprocessing(data):
  X = []
  y = []

  for x in data:
    cap = cv2.VideoCapture(x['video_path'])
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    idxs = np.linspace(0, total_frames - 1, 16, dtype = int)
    frames = []

    curr_idx = 0
    while cap.isOpened() and len(frames) < 16:
      ret, frame = cap.read()
      if not ret:
        break
      if curr_idx in idxs:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
      curr_idx += 1

    cap.release()

    if len(frames) == 16:
      proc_frames = processor(frames, return_tensors = 'pt')
      X.append(proc_frames.pixel_values.squeeze(0))
      y.append(x['label'])

  return {'pixel_values': torch.stack(X), 'labels' : torch.tensor(y)}


In [12]:
args = TrainingArguments(
    output_dir="./videomae_emovid_t4",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=3e-5,
    num_train_epochs=7,
    lr_scheduler_type="cosine",     
    warmup_steps=99,
    logging_steps=50,
    save_strategy="epoch",
    fp16=True,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    dataloader_pin_memory=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator = video_preprocessing
)

In [22]:
import os
import shutil

working_path = '/kaggle/working/videomae_emovid_t4/checkpoint-830'
input_path = '/kaggle/input/notebooks/timurbartia/notebookb9b058b77d/videomae_emovid_t4/checkpoint-1162'
if not os.path.exists(working_path):
    shutil.copytree(input_path, working_path)
    

In [7]:

# trainer.train(resume_from_checkpoint=working_path)

Step,Training Loss
500,1.263089
550,1.222264
600,1.222795
650,1.197065
700,0.884984
750,0.719009
800,0.683810
850,0.569258
900,0.389111
950,0.366240


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1162, training_loss=0.3668289504650333, metrics={'train_runtime': 23283.7753, 'train_samples_per_second': 3.188, 'train_steps_per_second': 0.05, 'total_flos': 9.249783153442475e+19, 'train_loss': 0.3668289504650333, 'epoch': 7.0})

In [13]:
test_dataset = Dataset.from_dict({
    "video_path": X_test,
    "label": y_test
})
label_feature = train_dataset.features["label"]

test_dataset = test_dataset.cast_column("label", label_feature)

Casting the dataset:   0%|          | 0/2651 [00:00<?, ? examples/s]

In [14]:
test_results = trainer.predict(test_dataset=test_dataset)
test_results.label_ids

array([3, 1, 4, ..., 7, 0, 5], shape=(2651,))

In [15]:
from sklearn.metrics import classification_report
pred = np.argmax(test_results.predictions, axis=-1)
print(classification_report(test_results.label_ids, pred))

              precision    recall  f1-score   support

           0       0.11      0.12      0.11       230
           1       0.21      0.10      0.13       580
           2       0.12      0.21      0.15       219
           3       0.05      0.05      0.05       270
           4       0.09      0.27      0.14       260
           5       0.09      0.07      0.08       300
           6       0.20      0.07      0.11       480
           7       0.14      0.12      0.13       312

    accuracy                           0.12      2651
   macro avg       0.12      0.13      0.11      2651
weighted avg       0.14      0.12      0.11      2651

